In [39]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("EU_AI_Act_2024.pdf")
pages = loader.load()

In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [45]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

In [46]:
chunks = splitter.split_documents(pages)

In [47]:
print(f"Number of chunks: {len(chunks)}")
print(chunks[0].page_content)

Number of chunks: 1455
REGUL A TION (EU) 2024/1689 OF THE EUR OPEAN P ARLIAMENT AND OF THE CO UNCIL
of 13 June 2024
laying do wn har monised r ules on ar tif icial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directiv es 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Ar tif icial Intelligence A ct)
(T ext with EEA relevance)
THE EUR OPEAN P ARLIAMENT AND THE COUNCIL OF THE EUR OPEAN UNION,


In [48]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
from langchain_community.vectorstores import Chroma

#stores the cunks inside the embedded model
vectorstore = Chroma.from_documents(chunks, embedding_model)

In [51]:
test_embedding = embedding_model.embed_query("EU AI regulations")
print(f"Length: {len(test_embedding)}")
print(test_embedding[:10])

Length: 384
[-0.01838821917772293, -0.012704242020845413, 0.010583501309156418, -0.07273206859827042, 0.05125313252210617, 0.031875453889369965, 0.01746824011206627, 0.01628563366830349, -0.040404122322797775, 0.02225891500711441]


In [54]:
results = vectorstore.similarity_search("requirements for high-risk AI systems?", k=3)
print(results[0].page_content)

of data.
(166) It is im por tant that AI systems relate d to products that are not high-r isk in accordance with this Regulation and thus 
are not required to comp ly with the requirements set out f or high-r isk AI systems are never theless safe when placed 
on the marke t or put into ser vice. T o contr ibut e to this objective, Regulation (EU) 2023/988 of the European 
P arliament and of the Council (
53
) would apply as a saf ety net.


In [55]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key="PASTE_API_KEY_HERE")

In [56]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

In [59]:
response = qa_chain.invoke({"query": "What are the transparency obligations for AI providers?"})
print(response["result"])

The provided context mentions that transparency obligations are laid down for certain AI systems. It also refers to "Article 50" for the practical implementation of these transparency obligations.

However, the specific details of what these transparency obligations entail for AI providers are not described in the given text.


In [58]:
response = qa_chain.invoke({"query": "What penalties can companies face for violating the AI Act?"})
print(response["result"])

Companies can face **administrative fines** for infringements of this Regulation.

Additionally, practices prohibited by other Union laws, such as data protection law, non-discrimination law, consumer protection law, and competition law, are not affected by this Regulation. Therefore, companies could also face penalties under those existing laws if their AI systems violate them.
